# Where should new childcare facilities go? Run the analysis for a city

This notebook runs the whole analysis from start to finish, for a small **demo city** or **your own city**.

**How to use it:** choose **Run → Run All Cells**. The notebook asks you a few questions, then you just wait. It reports what is happening the whole time, and the results appear at the bottom.

**What it does**

| Step | What happens | Do you need to do anything? |
|---|---|---|
| 1 | Checks your computer | No |
| 2 | Asks which city and how many new facilities (P), and checks the data | Answer two questions |
| 3 | Builds the model inputs | No |
| 4 | Asks for your Gurobi licence | Answer one or more questions |
| 5 | Solves the three models for your **P** | Wait |
| 6 | Shows the results | Read them |

**The number of new facilities P is yours to choose** (press Enter for the default, P = 3). A larger P takes longer to solve.

**Before a long run:** keep the computer awake and this notebook open. A city the size of Münster needs about 10 GB of free memory and 0.5-2 hours to solve. The demo city takes about a minute, so it is a good first try.

**The three models**

- **Utilitarian:** minimises the total walking time of all children (aggregate efficiency).
- **Individual Rawlsian:** protects the worst-served location in the priority group.
- **District Rawlsian:** protects the priority district whose average child walks longest.

The priority group, **Q4**, is the most vulnerable quarter of districts. Demand is **estimated potential demand**, not observed enrolment.

In [ ]:
# Optional: to skip the questions, set these before running. Leave as None to be asked.
CITY = None  # "demo", or the path of your city configuration
NEW_FACILITIES = None  # number of new facilities P (whole number >= 1); None asks, Enter keeps 3
LICENCE = None  # "installed", "file" or "wls"
LICENCE_FILE = None  # path of gurobi.lic, if LICENCE = "file"

## 1. Check this computer

In [ ]:
import shutil
import sys

try:
    from childcare_access import guided_run as guide
    from childcare_access.config import PROJECT_ROOT
except ImportError:
    raise SystemExit(
        "The project is not installed in this environment. Follow README.md, then restart this notebook."
    ) from None

free = shutil.disk_usage(PROJECT_ROOT).free / 1e9
print(f"Python {sys.version.split()[0]}; project folder {PROJECT_ROOT.name}; {free:.0f} GB of free disk space")

## 2. Choose a city and P, and check the data

- **Demo city:** a small synthetic city, created automatically.
- **Your own city:** the notebook creates the settings for you by asking questions one at a time. It reads your files and lets you pick the right columns from a list. **Have your data files ready on this computer first**; the list is in `README.md`. **Made a mistake?** Type `b` to go back one question (type it again to go further back). Before anything is saved you see a summary of your answers and can still go back. Type `q` to stop without saving.
- **Check a file before answering.** After you give the path of a file, type `2` to see its first rows as a table, or `3` to see it on a map (with your districts in grey). Press Enter to go on.
- **Long lists of values** (for example the building uses) show the most common values first. Type a word, such as `wohn`, to see only the values that contain it. Pick several at once with `2,5` or `3-6`, or take every match with `all with wohn`.
- **Prefer to fill in a file yourself?** Choose *Your own city*, then option 3. You get a sample settings file (`config/<city>.toml`) with a link that opens it here. Fill in every value marked `<-- REPLACE`, save, and run the cell again with its path.
- Your settings are saved as a file, so next time you only give its path.

In [ ]:
city = guide.choose_city(CITY)
p = guide.choose_p(NEW_FACILITIES)
guide.check_data(city)

## 3. Build the model inputs

This turns the raw data into demand locations, walking times and candidate sites. It needs neither Gurobi nor an internet connection.

In [ ]:
guide.build_inputs(city)

## 4. Your Gurobi licence

The models are solved with [Gurobi](https://www.gurobi.com/), which needs a licence. Academic licences are free. The next cell explains how your licence is handled before it asks for anything.

In [ ]:
licence = guide.ask_licence(LICENCE, LICENCE_FILE)
guide.check_licence(city, licence)

## 5. Solve the three models

Now you can wait. Every message appears below as it happens, and a *still working* line appears when the solver is busy, with its latest progress.

Stopping at a time limit is normal. With the default settings, Stage 2 of the Individual Rawlsian model often stops at its limit. The results say which solutions are proven optimal.

In [ ]:
run = guide.solve(city, licence, p)

## 6. Results

In [ ]:
results = guide.summarise(city, run)
guide.show_results(results)

**How to read these results**

- Differences between the three models here come from their objectives, because they share the same inputs.
- Only *proven optimal* results are optimal. A *time-limited* result is a feasible solution, not a proven optimum.
- Sites may differ between runs where several solutions are equally good, or where a stage is time-limited.
- Results describe potential walking access. Kita capacity, opening hours and parental choice are not modelled, and candidate sites are not checked for availability.
- For a full worked example with a real city, see the Münster case study in `muenster_example.ipynb`.